# 🎨 VELARA 3D ARTIST — Image → 3D (TripoSR)

**Kaam:** tum image doge → AI 3D model banayega → game-ready GLB (≤20k tris) → repo mein `review` queue ke saath push.
Phir **DM preview chat mein dikhayega — USER APPROVE karega tabhi game mein jayega.**

**Setup (ek baar):** Secrets mein `GHTOKEN` daalo (fine-grained PAT, sirf repo `priyanshagrahari54-blip/Velara`, Contents Read+Write).
- **Colab:** 🔑 sidebar → Secrets → `GHTOKEN`
- **Kaggle:** Add-ons → Secrets → Label `GHTOKEN`

**Image dena:** Cell 3 mein `IMAGE_URL` link daalo, YA file upload karo (Colab: files panel; Kaggle: dataset ya `/kaggle/input/...` path).


In [ ]:
# 1) ENV CHECK + INSTALL (~3 min, ek baar)
import subprocess, sys, os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ NAHI — GPU instance chuno!')
assert torch.cuda.is_available(), 'GPU on (T4/P100), phir Run-All'

!git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR 2>/dev/null || !git clone -q https://github.com/VAST-AI-Research/TripoSR.git ./TripoSR
TRIPOSR = '/content/TripoSR' if os.path.exists('/content/TripoSR') else './TripoSR'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{TRIPOSR}/requirements.txt'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rembg', 'onnxruntime', 'fast-simplification', 'huggingface-hub'], check=False)
sys.path.insert(0, TRIPOSR)
os.environ['HF_HOME'] = '/tmp/hf'
print('install done')

In [ ]:
# 2) MODEL LOAD (~1 min, weights ~1.6GB download)
from tsr.system import TSR
model = TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
model.renderer.set_chunk_size(8192)
model.to('cuda')
print('TripoSR ready ✅')

In [ ]:
# 3) ⬇️ APNI IMAGE YAHAN DO — URL ya upload path
IMAGE_URL = ''            # option A: 'https://.../photo.jpg'
UPLOAD_PATH = ''          # option B: '/content/photo.png' / '/kaggle/input/dataset/photo.jpg'
ASSET_NAME = 'my-asset'   # id: factory/out/<id>.glb banega
MAX_TRIS = 20000          # game budget

In [ ]:
# 4) IMAGE → 3D → GAME-READY GLB
import urllib.request, torch
from PIL import Image
from tsr.utils import remove_background, resize_foreground
import rembg, numpy as np, trimesh

src = IMAGE_URL or UPLOAD_PATH
assert src, 'Cell 3 mein IMAGE_URL ya UPLOAD_PATH bharo!'
if IMAGE_URL:
    urllib.request.urlretrieve(IMAGE_URL, '/tmp/input.png'); src = '/tmp/input.png'

img = remove_background(Image.open(src).convert('RGB'), rembg.new_session())
img = resize_foreground(img, 0.9)
img = img.convert('RGBA'); a = np.array(img).astype(np.float32)/255.0
img = Image.fromarray(((a[:,:,:3]*a[:,:,3:4] + (1-a[:,:,3:4])*0.5)*255).astype(np.uint8))

with torch.no_grad():
    scene_codes = model([img], device='cuda')
mesh = model.extract_mesh(scene_codes, has_vertex_color=True, resolution=256)[0]
print('raw mesh:', len(mesh.vertices), 'verts /', len(mesh.faces), 'tris')

# decimate — game budget
if len(mesh.faces) > MAX_TRIS:
    try:
        import fast_simplification
        v, f = fast_simplification.simplify(mesh.vertices, mesh.faces, target_count=MAX_TRIS)
        mesh = trimesh.Trimesh(v, f, process=False)
        print('decimated ->', len(mesh.faces), 'tris')
    except Exception as e:
        print('⚠ decimation fail (DM Blender se karega):', e)

os.makedirs('/tmp/out', exist_ok=True)
GLB = f'/tmp/out/{ASSET_NAME}.glb'
mesh.export(GLB)
print('GLB:', GLB, os.path.getsize(GLB)//1024, 'KB')

In [ ]:
# 5) PREVIEW RENDER (chat approval ke liye DM use karega)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
m = mesh
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_trisurf(m.vertices[:,0], m.vertices[:,2], m.vertices[:,1],
                triangles=m.faces, cmap='viridis', linewidth=0)
ax.set_axis_off(); ax.view_init(elev=15, azim=-60)
plt.tight_layout(); plt.savefig(f'/tmp/out/{ASSET_NAME}_preview.png', dpi=110)
print('preview saved')
try:
    from google.colab import files
    files.download(GLB)
except Exception:
    pass

In [ ]:
# 6) REPO PUSH (GHTOKEN secret se) — review queue ke saath
import json, subprocess, os

TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GHTOKEN')
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret('GHTOKEN')
    except Exception:
        pass

if not TOKEN:
    print('⚠ GHTOKEN nahi mila — GLB manually download karke DM ko dena (files panel se).')
else:
    REPO = 'https://x-access-token:' + TOKEN + '@github.com/priyanshagrahari54-blip/Velara.git'
    subprocess.run(['rm', '-rf', '/tmp/velara'], check=False)
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, '/tmp/velara'], check=True)
    os.makedirs('/tmp/velara/factory/out', exist_ok=True)
    subprocess.run(['cp', GLB, f'/tmp/velara/factory/out/{ASSET_NAME}.glb'], check=True)
    subprocess.run(['cp', f'/tmp/out/{ASSET_NAME}_preview.png', f'/tmp/velara/factory/out/{ASSET_NAME}_preview.png'], check=False)
    job = {'id': ASSET_NAME, 'script': 'artist-triposr (image23d)', 'args': [],
           'brief': 'Image->3D via TripoSR — USER REVIEW PENDING',
           'status': 'review', 'priority': 2, 'worker': 'artist-gpu'}
    json.dump(job, open(f'/tmp/velara/factory/queue/{ASSET_NAME}.json', 'w'), indent=1)
    subprocess.run(['git', '-C', '/tmp/velara', 'config', 'user.name', 'artist-bot'], check=True)
    subprocess.run(['git', '-C', '/tmp/velara', 'config', 'user.email', 'artist@velara'], check=True)
    subprocess.run(['git', '-C', '/tmp/velara', 'add', 'factory/out', 'factory/queue'], check=True)
    subprocess.run(['git', '-C', '/tmp/velara', 'commit', '-q', '-m',
                    f'artist(image23d): {ASSET_NAME} review ke liye [skip ci]'], check=True)
    subprocess.run(['git', '-C', '/tmp/velara', 'push', '-q', 'origin', 'main'], check=True)
    print('✅ PUSHED — repo mein review status. DM (assistant) agle turn mein preview dikha ke approval lega.')